# Bronze Layer

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

## Step 1: Import Required Libraries

In [0]:
spark

## Step 2: Define File Paths

In [0]:
bronze_path = "/Volumes/learntrack_lms_analytics/default/bronze_layer"

learners_file = f"{bronze_path}/learners.csv"
courses_file = f"{bronze_path}/courses.csv"
enrolment_file = f"{bronze_path}/enrolment_activity.csv"

## Step 3: Read the CSV Files

In [0]:
learners_df = (
    spark.read
         .option("header", True)
         .option("inferSchema", True)
         .csv(learners_file)
)

courses_df = (
    spark.read
         .option("header", True)
         .option("inferSchema", True)
         .csv(courses_file)
)

enrolment_df = (
    spark.read
         .option("header", True)
         .option("inferSchema", True)
         .csv(enrolment_file)
)

## Step 4: Verify the Data

In [0]:
learners_df.show(5, truncate=False)


+----------+----------------+------------------------------+------------+-------+-----------------+-----------------+
|learner_id|learner_name    |email                         |phone_number|city   |registration_date|subscription_type|
+----------+----------------+------------------------------+------------+-------+-----------------+-----------------+
|LRN0001   |Ananya Sharma   |ananya.sharma.1@gmail.com     |9433218196  |Mumbai |2022-04-06       |Free             |
|LRN0002   |Sachin Pillai   |sachin.pillai.2@gmail.com     |9083863794  |Indore |2022-06-13       |Premium          |
|LRN0003   |Suresh Mishra   |suresh.mishra.3@rediffmail.com|9235116155  |Chennai|2022-02-14       |Premium          |
|LRN0004   |Vijay Kumar     |vijay.kumar.4@gmail.com       |9618495931  |Surat  |2022-08-22       |Premium          |
|LRN0005   |Priya Chatterjee|priya.chatterjee.5@yahoo.com  |9316475255  |Surat  |2022-10-01       |Premium          |
+----------+----------------+---------------------------

In [0]:
courses_df.show(5, truncate=False)


+---------+-----------------------------+---------------+-------------+---------------+--------------+----------------+---------+
|course_id|course_title                 |category       |instructor_id|instructor_name|duration_hours|difficulty_level|price_inr|
+---------+-----------------------------+---------------+-------------+---------------+--------------+----------------+---------+
|CRS001   |Python for Data Science      |Data Science   |INS009       |Suresh Gupta   |15            |Intermediate    |2986     |
|CRS002   |Machine Learning Fundamentals|AI & ML        |INS006       |Priya Singh    |10            |Beginner        |223      |
|CRS003   |Deep Learning with TensorFlow|AI & ML        |INS004       |Sunita Rao     |10            |Advanced        |13612    |
|CRS004   |React.js Complete Guide      |Web Development|INS014       |Pooja Mishra   |30            |Advanced        |6867     |
|CRS005   |Node.js Backend Development  |Web Development|INS007       |Vikas Mehta    |20 

In [0]:
enrolment_df.show(5, truncate=False)

+------------+----------+---------+----------+------------------------+----------------------+---------+------------+------------------+----------------+--------+---------------+------------------+
|enrolment_id|learner_id|course_id|enrol_date|expected_completion_date|actual_completion_date|status   |progress_pct|last_activity_date|assessment_score|attempts|feedback_rating|certificate_issued|
+------------+----------+---------+----------+------------------------+----------------------+---------+------------+------------------+----------------+--------+---------------+------------------+
|ENR00460    |LRN0376   |CRS001   |2024-01-13|2024-02-12              |2024-02-21            |Completed|100         |2024-02-21        |79.22           |1       |2              |Yes               |
|ENR00681    |LRN0486   |CRS055   |2024-01-16|2024-01-26              |2024-01-19            |Completed|100         |2024-01-19        |56.62           |1       |3              |No                |
|ENR01477 

## Step 5: Check the Schema

In [0]:
learners_df.printSchema()



root
 |-- learner_id: string (nullable = true)
 |-- learner_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone_number: long (nullable = true)
 |-- city: string (nullable = true)
 |-- registration_date: date (nullable = true)
 |-- subscription_type: string (nullable = true)



In [0]:
courses_df.printSchema()


root
 |-- course_id: string (nullable = true)
 |-- course_title: string (nullable = true)
 |-- category: string (nullable = true)
 |-- instructor_id: string (nullable = true)
 |-- instructor_name: string (nullable = true)
 |-- duration_hours: integer (nullable = true)
 |-- difficulty_level: string (nullable = true)
 |-- price_inr: integer (nullable = true)



In [0]:

enrolment_df.printSchema()

root
 |-- enrolment_id: string (nullable = true)
 |-- learner_id: string (nullable = true)
 |-- course_id: string (nullable = true)
 |-- enrol_date: date (nullable = true)
 |-- expected_completion_date: date (nullable = true)
 |-- actual_completion_date: date (nullable = true)
 |-- status: string (nullable = true)
 |-- progress_pct: integer (nullable = true)
 |-- last_activity_date: date (nullable = true)
 |-- assessment_score: double (nullable = true)
 |-- attempts: integer (nullable = true)
 |-- feedback_rating: integer (nullable = true)
 |-- certificate_issued: string (nullable = true)



## Step 6: Count the Records

In [0]:
print("Learners :", learners_df.count())


Learners : 500


In [0]:
print("Courses :", courses_df.count())


Courses : 60


In [0]:
print("Enrolments :", enrolment_df.count())

Enrolments : 2000


## Step 7: Write Bronze Tables as Delta

In [0]:
learners_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("learntrack_lms_analytics.default.bronze_learners")

In [0]:
courses_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("learntrack_lms_analytics.default.bronze_courses")

In [0]:
enrolment_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("learntrack_lms_analytics.default.bronze_enrolment_activity")

## Step 8: Verify the Bronze Tables

In [0]:
(spark.table("learntrack_lms_analytics.default.bronze_learners")).show(10)

+----------+----------------+--------------------+------------+----------+-----------------+-----------------+
|learner_id|    learner_name|               email|phone_number|      city|registration_date|subscription_type|
+----------+----------------+--------------------+------------+----------+-----------------+-----------------+
|   LRN0001|   Ananya Sharma|ananya.sharma.1@g...|  9433218196|    Mumbai|       2022-04-06|             Free|
|   LRN0002|   Sachin Pillai|sachin.pillai.2@g...|  9083863794|    Indore|       2022-06-13|          Premium|
|   LRN0003|   Suresh Mishra|suresh.mishra.3@r...|  9235116155|   Chennai|       2022-02-14|          Premium|
|   LRN0004|     Vijay Kumar|vijay.kumar.4@gma...|  9618495931|     Surat|       2022-08-22|          Premium|
|   LRN0005|Priya Chatterjee|priya.chatterjee....|  9316475255|     Surat|       2022-10-01|          Premium|
|   LRN0006|    Karan Pillai|karan.pillai.6@ou...|  9283276483|    Bhopal|       2022-02-27|             Free|
|

In [0]:
(spark.table("learntrack_lms_analytics.default.bronze_courses")).show()


+---------+--------------------+------------------+-------------+---------------+--------------+----------------+---------+
|course_id|        course_title|          category|instructor_id|instructor_name|duration_hours|difficulty_level|price_inr|
+---------+--------------------+------------------+-------------+---------------+--------------+----------------+---------+
|   CRS001|Python for Data S...|      Data Science|       INS009|   Suresh Gupta|            15|    Intermediate|     2986|
|   CRS002|Machine Learning ...|           AI & ML|       INS006|    Priya Singh|            10|        Beginner|      223|
|   CRS003|Deep Learning wit...|           AI & ML|       INS004|     Sunita Rao|            10|        Advanced|    13612|
|   CRS004|React.js Complete...|   Web Development|       INS014|   Pooja Mishra|            30|        Advanced|     6867|
|   CRS005|Node.js Backend D...|   Web Development|       INS007|    Vikas Mehta|            20|        Beginner|      119|
|   CRS0

In [0]:

(spark.table("learntrack_lms_analytics.default.bronze_enrolment_activity")).show()

+------------+----------+---------+----------+------------------------+----------------------+-----------+------------+------------------+----------------+--------+---------------+------------------+
|enrolment_id|learner_id|course_id|enrol_date|expected_completion_date|actual_completion_date|     status|progress_pct|last_activity_date|assessment_score|attempts|feedback_rating|certificate_issued|
+------------+----------+---------+----------+------------------------+----------------------+-----------+------------+------------------+----------------+--------+---------------+------------------+
|    ENR00460|   LRN0376|   CRS001|2024-01-13|              2024-02-12|            2024-02-21|  Completed|         100|        2024-02-21|           79.22|       1|              2|               Yes|
|    ENR00681|   LRN0486|   CRS055|2024-01-16|              2024-01-26|            2024-01-19|  Completed|         100|        2024-01-19|           56.62|       1|              3|                No|
